# Pokemon Pipeline — Exploración DuckDB
### Python para Data Engineers · DataHackers Academy
---
**Objetivo:** Explorar las tablas cargadas en DuckDB después de correr `pipeline.py`.

Requisito: haber ejecutado `python pipeline.py` al menos una vez.

## 0. Conexión a DuckDB

In [39]:
import warnings
import duckdb
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DB_PATH = '../data/processed/pokemon.duckdb'
con = duckdb.connect(DB_PATH)

print(f'Conectado a: {DB_PATH}')

Conectado a: ../data/processed/pokemon.duckdb


In [40]:
# Tablas disponibles
con.execute('SHOW TABLES').df()

,name
0,dim_pokemon
1,fact_battles
2,rejected_records
3,tabla_comparacion


## 1. dim_pokemon — 150 Pokemon con stats base y columnas derivadas

In [41]:
# Schema de la tabla
con.execute('DESCRIBE dim_pokemon').df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,type1,VARCHAR,YES,None,None,None
3,type2,VARCHAR,YES,None,None,None
4,hp,BIGINT,YES,None,None,None
5,attack,BIGINT,YES,None,None,None
6,defense,BIGINT,YES,None,None,None
7,special_attack,BIGINT,YES,None,None,None
8,special_defense,BIGINT,YES,None,None,None
9,speed,BIGINT,YES,None,None,None


In [42]:
# Primeras filas
con.execute("""
    SELECT id, name, type1, type2, stat_role, total_stats,
           hp, attack, defense, special_attack, special_defense, speed,
           height_m, weight_kg
    FROM dim_pokemon
    ORDER BY id
    LIMIT 10
""").df()

,id,name,type1,type2,stat_role,total_stats,hp,attack,defense,special_attack,special_defense,speed,height_m,weight_kg
0,1,bulbasaur,grass,poison,special_sweeper,318,45,49,49,65,65,45,0.70,6.90
1,2,ivysaur,grass,poison,special_sweeper,405,60,62,63,80,80,60,1.00,13.00
2,3,venusaur,grass,poison,special_sweeper,525,80,82,83,100,100,80,2.00,100.00
3,4,charmander,fire,none,speedster,309,39,52,43,60,50,65,0.60,8.50
4,5,charmeleon,fire,none,special_sweeper,405,58,64,58,80,65,80,1.10,19.00
5,6,charizard,fire,flying,special_sweeper,534,78,84,78,109,85,100,1.70,90.50
6,7,squirtle,water,none,tank,314,44,48,65,50,64,43,0.50,9.00
7,8,wartortle,water,none,tank,405,59,63,80,65,80,58,1.00,22.50
8,9,blastoise,water,none,tank,530,79,83,100,85,105,78,1.60,85.50
9,10,caterpie,bug,none,tank,195,45,30,35,20,20,45,0.30,2.90


In [43]:
# Top 10 por total_stats
con.execute("""
    SELECT name, type1, type2, stat_role, total_stats,
           hp, attack, defense, special_attack, special_defense, speed
    FROM dim_pokemon
    ORDER BY total_stats DESC
    LIMIT 10
""").df()

,name,type1,type2,stat_role,total_stats,hp,attack,defense,special_attack,special_defense,speed
0,mewtwo,psychic,none,special_sweeper,680,106,110,90,154,90,130
1,dragonite,dragon,flying,physical_sweeper,600,91,134,95,100,100,80
2,articuno,ice,flying,tank,580,90,85,100,95,125,85
3,zapdos,electric,flying,special_sweeper,580,90,90,85,125,90,100
4,moltres,fire,flying,special_sweeper,580,90,100,90,125,85,90
5,arcanine,fire,none,physical_sweeper,555,90,110,80,100,80,95
6,gyarados,water,flying,physical_sweeper,540,95,125,79,60,100,81
7,snorlax,normal,none,tank,540,160,110,65,65,110,30
8,lapras,water,ice,tank,535,130,85,80,85,95,60
9,charizard,fire,flying,special_sweeper,534,78,84,78,109,85,100


In [44]:
# Stats promedio por tipo primario
con.execute("""
    SELECT type1,
           COUNT(*)                    AS cantidad,
           ROUND(AVG(hp), 1)           AS avg_hp,
           ROUND(AVG(attack), 1)       AS avg_attack,
           ROUND(AVG(defense), 1)      AS avg_defense,
           ROUND(AVG(special_attack), 1)  AS avg_sp_atk,
           ROUND(AVG(special_defense), 1) AS avg_sp_def,
           ROUND(AVG(speed), 1)        AS avg_speed,
           ROUND(AVG(total_stats), 1)  AS avg_total
    FROM dim_pokemon
    GROUP BY type1
    ORDER BY avg_total DESC
""").df()

,type1,cantidad,avg_hp,avg_attack,avg_defense,avg_sp_atk,avg_sp_def,avg_speed,avg_total
0,ice,2,77.50,67.50,67.50,105.00,110.00,90.00,517.50
1,fire,12,63.80,83.90,62.60,84.60,76.70,84.00,455.60
2,psychic,7,58.70,54.40,51.40,104.30,90.70,92.00,451.60
3,electric,9,54.40,62.00,64.70,91.10,73.30,100.00,445.60
4,dragon,3,64.30,94.00,68.30,73.30,73.30,66.70,440.00
5,rock,9,53.90,82.20,110.00,60.60,55.60,58.30,420.60
6,fighting,7,63.60,102.90,61.00,45.00,73.60,66.10,412.10
7,water,28,64.50,70.30,77.50,64.80,66.40,67.70,411.20
8,grass,12,65.00,70.70,69.60,87.90,65.00,52.10,410.30
9,ghost,3,45.00,50.00,45.00,115.00,55.00,95.00,405.00


In [45]:
# Distribucion de stat_role
con.execute("""
    SELECT stat_role,
           COUNT(*) AS cantidad,
           ROUND(AVG(total_stats), 1) AS avg_total_stats
    FROM dim_pokemon
    GROUP BY stat_role
    ORDER BY cantidad DESC
""").df()

,stat_role,cantidad,avg_total_stats
0,tank,56,393.00
1,physical_sweeper,37,421.20
2,speedster,30,379.10
3,special_sweeper,27,444.00


## 2. fact_battles — Enfrentamientos enriquecidos con stats al nivel

In [46]:
# Primeras filas — columnas clave
con.execute("""
    SELECT pokemon_a, a_name, nivel_a, a_total_lv,
           pokemon_b, b_name, nivel_b, b_total_lv,
           ganador, poder_total, diferencia_poder, ventaja_pct
    FROM fact_battles
    LIMIT 10
""").df()

,pokemon_a,a_name,nivel_a,a_total_lv,pokemon_b,b_name,nivel_b,b_total_lv,ganador,poder_total,diferencia_poder,ventaja_pct
0,7,squirtle,58,455,140,kabuto,1,40,squirtle,495,415,1037.50
1,87,dewgong,45,507,98,krabby,6,78,dewgong,585,429,550.00
2,93,haunter,30,308,18,pidgeot,11,149,haunter,457,159,106.70
3,42,golbat,90,944,54,psyduck,83,647,golbat,1591,297,45.90
4,70,weepinbell,42,402,143,snorlax,100,1215,snorlax,1617,813,202.20
5,146,moltres,64,840,81,magnemite,83,655,moltres,1495,185,28.20
6,150,mewtwo,29,456,150,mewtwo,18,295,mewtwo,751,161,54.60
7,88,grimer,1,40,112,rhydon,93,1027,rhydon,1067,987,2467.50
8,29,nidoran-f,31,234,79,slowpoke,11,112,nidoran-f,346,122,108.90
9,33,nidorino,68,595,141,kabutops,55,633,kabutops,1228,38,6.40


In [47]:
# Distribucion de resultados
con.execute("""
    SELECT
        CASE
            WHEN ganador = a_name THEN 'gana_a'
            WHEN ganador = b_name THEN 'gana_b'
            ELSE 'empate'
        END AS resultado,
        COUNT(*) AS cantidad,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM fact_battles
    GROUP BY 1
    ORDER BY cantidad DESC
""").df()

,resultado,cantidad,pct
0,gana_a,194,51.50
1,gana_b,181,48.00
2,empate,2,0.50


In [48]:
# Top 10 Pokemon con mas victorias (como A o B)
con.execute("""
    SELECT name,
           SUM(total_battles) AS batallas,
           SUM(wins)          AS victorias,
           ROUND(SUM(wins) * 100.0 / SUM(total_battles), 1) AS win_pct
    FROM (
        SELECT a_name AS name, COUNT(*) AS total_battles,
               SUM(CASE WHEN ganador = a_name THEN 1 ELSE 0 END) AS wins
        FROM fact_battles GROUP BY a_name
        UNION ALL
        SELECT b_name, COUNT(*),
               SUM(CASE WHEN ganador = b_name THEN 1 ELSE 0 END)
        FROM fact_battles GROUP BY b_name
    )
    GROUP BY name
    HAVING batallas >= 5
    ORDER BY victorias DESC
    LIMIT 10
""").df()

,name,batallas,victorias,win_pct
0,zapdos,11.00,8.00,72.70
1,kabutops,9.00,8.00,88.90
2,mewtwo,8.00,7.00,87.50
3,arbok,9.00,7.00,77.80
4,aerodactyl,7.00,7.00,100.00
5,raichu,7.00,6.00,85.70
6,dragonair,7.00,6.00,85.70
7,electrode,6.00,6.00,100.00
8,krabby,12.00,6.00,50.00
9,cloyster,8.00,6.00,75.00


In [49]:
# Win rate por tipo primario
con.execute("""
    SELECT p.type1,
           COUNT(DISTINCT b.name)                                    AS pokemon_count,
           ROUND(AVG(p.total_stats), 1)                              AS avg_total_stats,
           SUM(b.wins)                                               AS total_wins,
           SUM(b.total_battles)                                      AS total_battles,
           ROUND(SUM(b.wins) * 100.0 / SUM(b.total_battles), 1)     AS win_pct
    FROM (
        SELECT a_name AS name,
               SUM(CASE WHEN ganador = a_name THEN 1 ELSE 0 END) AS wins,
               COUNT(*) AS total_battles
        FROM fact_battles GROUP BY a_name
        UNION ALL
        SELECT b_name,
               SUM(CASE WHEN ganador = b_name THEN 1 ELSE 0 END),
               COUNT(*)
        FROM fact_battles GROUP BY b_name
    ) b
    JOIN dim_pokemon p ON p.name = b.name
    GROUP BY p.type1
    ORDER BY win_pct DESC
""").df()

,type1,pokemon_count,avg_total_stats,total_wins,total_battles,win_pct
0,dragon,3,408.00,13.00,17.00,76.50
1,ice,2,517.50,7.00,10.00,70.00
2,electric,9,445.60,39.00,60.00,65.00
3,rock,9,420.60,31.00,51.00,60.80
4,psychic,7,444.90,21.00,35.00,60.00
5,water,28,412.70,75.00,145.00,51.70
6,poison,14,396.30,31.00,60.00,51.70
7,normal,22,381.70,56.00,110.00,50.90
8,fire,12,454.30,23.00,47.00,48.90
9,fighting,7,430.00,15.00,32.00,46.90


In [50]:
# Batallas mas parejas (menor diferencia de poder)
con.execute("""
    SELECT a_name, nivel_a, a_total_lv,
           b_name, nivel_b, b_total_lv,
           ganador, diferencia_poder, ventaja_pct
    FROM fact_battles
    WHERE ganador != 'empate'
    ORDER BY diferencia_poder ASC
    LIMIT 10
""").df()

,a_name,nivel_a,a_total_lv,b_name,nivel_b,b_total_lv,ganador,diferencia_poder,ventaja_pct
0,clefairy,26,226,goldeen,26,225,clefairy,1,0.40
1,magneton,67,722,horsea,100,725,horsea,3,0.40
2,victreebel,59,669,hypno,60,673,hypno,4,0.60
3,dugtrio,81,803,aerodactyl,69,811,aerodactyl,8,1.00
4,wigglytuff,43,450,pidgeotto,54,464,pidgeotto,14,3.10
5,grimer,19,176,farfetchd,19,194,farfetchd,18,10.20
6,doduo,56,437,hitmonlee,42,457,hitmonlee,20,4.60
7,ekans,17,146,lapras,8,125,ekans,21,16.80
8,dragonite,46,631,seel,83,654,seel,23,3.60
9,onix,26,259,kabuto,25,236,onix,23,9.70


## 3. tabla_comparacion — Vista legible del enfrentamiento

In [51]:
# Primeras filas
con.execute("""
    SELECT pokemon, tipo1, nivel, poder,
           rival, rival_tipo1, rival_nivel, rival_poder,
           ganador, resultado, diferencia_poder, ventaja_pct
    FROM tabla_comparacion
    LIMIT 10
""").df()

,pokemon,tipo1,nivel,poder,rival,rival_tipo1,rival_nivel,rival_poder,ganador,resultado,diferencia_poder,ventaja_pct
0,squirtle,water,58,455,kabuto,rock,1,40,squirtle,gana,415,1037.50
1,dewgong,water,45,507,krabby,water,6,78,dewgong,gana,429,550.00
2,haunter,ghost,30,308,pidgeot,normal,11,149,haunter,gana,159,106.70
3,golbat,poison,90,944,psyduck,water,83,647,golbat,gana,297,45.90
4,weepinbell,grass,42,402,snorlax,normal,100,1215,snorlax,pierde,813,202.20
5,moltres,fire,64,840,magnemite,electric,83,655,moltres,gana,185,28.20
6,mewtwo,psychic,29,456,mewtwo,psychic,18,295,mewtwo,gana,161,54.60
7,grimer,poison,1,40,rhydon,ground,93,1027,rhydon,pierde,987,2467.50
8,nidoran-f,poison,31,234,slowpoke,water,11,112,nidoran-f,gana,122,108.90
9,nidorino,poison,68,595,kabutops,rock,55,633,kabutops,pierde,38,6.40


In [52]:
# Conteo de resultados
con.execute("""
    SELECT resultado,
           COUNT(*) AS cantidad,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM tabla_comparacion
    GROUP BY resultado
    ORDER BY cantidad DESC
""").df()

,resultado,cantidad,pct
0,gana,194,51.50
1,pierde,181,48.00
2,empate,2,0.50


In [53]:
# Buscar enfrentamientos de un Pokemon especifico
POKEMON = 'pikachu'  # <- cambia aqui

con.execute(f"""
    SELECT pokemon, nivel, poder,
           rival, rival_tipo1, rival_nivel, rival_poder,
           resultado, ventaja_pct
    FROM tabla_comparacion
    WHERE pokemon = '{POKEMON}'
    ORDER BY resultado, ventaja_pct DESC
""").df()

,pokemon,nivel,poder,rival,rival_tipo1,rival_nivel,rival_poder,resultado,ventaja_pct
0,pikachu,69,544,caterpie,bug,16,111,gana,390.10
1,pikachu,56,447,cloyster,water,84,998,pierde,123.30
2,pikachu,74,581,ponyta,fire,100,955,pierde,64.40
3,pikachu,93,722,kingler,water,78,852,pierde,18.00


## 4. rejected_records — Registros descartados por el pipeline

In [54]:
# Total de rechazados y motivos
print(f'Total rechazados: {con.execute("SELECT COUNT(*) FROM rejected_records").fetchone()[0]}')
print()
con.execute("""
    SELECT motivo_rechazo,
           COUNT(*) AS cantidad
    FROM rejected_records
    GROUP BY motivo_rechazo
    ORDER BY cantidad DESC
""").df()

Total rechazados: 133



,motivo_rechazo,cantidad
0,pokemon_a nulo,17
1,pokemon_b nulo,11
2,duplicado exacto,10
3,pokemon_b fuera de rango (200.0),8
4,nivel_b fuera de rango (101.0),6
5,nivel_a nulo,6
6,nivel_a fuera de rango (-1.0),5
7,nivel_b fuera de rango (0.0),5
8,pokemon_b fuera de rango (-1.0),5
9,nivel_a fuera de rango (0.0),5


In [55]:
# Ejemplos de cada tipo de rechazo
con.execute("""
    SELECT *
    FROM rejected_records
    ORDER BY motivo_rechazo
    LIMIT 15
""").df()

,pokemon_a,nivel_a,pokemon_b,nivel_b,motivo_rechazo
0,77.0,69.0,130.0,73.0,duplicado exacto
1,85.0,84.0,111.0,18.0,duplicado exacto
2,92.0,46.0,64.0,53.0,duplicado exacto
3,64.0,58.0,88.0,18.0,duplicado exacto
4,25.0,56.0,91.0,84.0,duplicado exacto
5,132.0,72.0,11.0,21.0,duplicado exacto
6,55.0,0.0,87.0,54.0,duplicado exacto
7,26.0,41.0,-1.0,150.0,duplicado exacto
8,8.0,56.0,43.0,13.0,duplicado exacto
9,106.0,82.0,78.0,101.0,duplicado exacto


## 5. SQL libre — escribe tus propias queries

In [56]:
# Edita esta query para explorar lo que quieras
query = """
    SELECT p.name, p.type1, p.type2, p.total_stats,
           COUNT(fb.ganador) AS victorias
    FROM dim_pokemon p
    LEFT JOIN fact_battles fb
        ON fb.ganador = p.name
    GROUP BY p.name, p.type1, p.type2, p.total_stats
    ORDER BY victorias DESC
    LIMIT 15
"""

con.execute(query).df()

,name,type1,type2,total_stats,victorias
0,zapdos,electric,flying,580,8
1,arbok,poison,none,448,7
2,aerodactyl,rock,flying,515,7
3,kabutops,rock,water,495,7
4,electrode,electric,none,490,6
5,krabby,water,none,325,6
6,kadabra,psychic,none,400,6
7,mewtwo,psychic,none,680,6
8,cloyster,water,ice,525,6
9,wartortle,water,none,405,6


In [57]:
# Cerrar conexion al terminar
con.close()
print('Conexion cerrada')

Conexion cerrada
